## Tucson, AZ Small County

In [424]:
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz
import re

In [425]:
pr = pd.read_csv('raw_data/propertyradar.csv')

pr.to_csv("export_data/pr_import.csv", index=False)

In [426]:
#Create full address for record name

pr["Last Name"] = (
    pr["Address"].str.title() + ", " +
    pr["City"].str.title() + ", " +
    pr["State"].astype(str) + " " +
    pr["ZIP"].astype(str).str.replace(r"\.0$", "", regex=True)
)

In [427]:
#Create unique identifier

#Rename original APN column
pr.rename(columns={"APN": "Tax Id"}, inplace=True)

#Format unique identifier
def format_parcel(row):
    county = str(row["County"]).strip().title()
    parcel = str(row["Tax Id"]).strip()

    if county == "Maricopa":
        if len(parcel) == 10:
            return parcel
        else:
            return parcel[:10] + parcel[-1:]

    elif county == "Pinal":
        if len(parcel) == 11:
            return f"PINAL_AZ.{parcel[:3]}-{parcel[4:6]}-{parcel[-4:]}"
        else:
            return f"PINAL_AZ.{parcel[:3]}-{parcel[4:6]}-{parcel[-3:]}0"

    elif county in [
        "Coconino", "Cochise", "Graham", "Gila",
        "Mohave", "Santa Cruz", "Yavapai", "Yuma"
    ]:
        return f"{county.upper()}_AZ.{parcel}"

    elif county == "Pima":
        return f"PIMA_AZ.{parcel[:3]}-{parcel[4:6]}-{parcel[-4:]}"

    else:
        return parcel


#Create new APN column with formatting
pr["APN"] = pr.apply(format_parcel, axis=1)

In [428]:
#Format County

pr.rename(columns={"County": "oldCounty"}, inplace=True)

pr["County"] = pr["oldCounty"].str.title() + " - AZ"

In [429]:
#Format Market

pr["Market"] = np.where(
    pr["oldCounty"].str.strip().str.title().isin(["Maricopa", "Pinal"]),
    "Phoenix",
    np.where(
        pr["oldCounty"].str.strip().str.title() == "Pima",
        "Tucson",
        "AZ Small County"
    )
)

In [430]:
#Format Owner and Trustor

pr.rename(columns={"Owner": "Property Owner Name"}, inplace=True)

pr["Trustor"] = pr["Property Owner Name"].copy()

In [431]:
#Format Pool

pr.rename(columns={"Pool?": "notPool"}, inplace=True)

pr["Pool"] = np.where(pr["notPool"] == 0, "No", "Yes")

In [432]:
#Format NS Lien Position

pr["NS Lien Position"] = pr["FCL Loan Position"].astype(str).str.strip().map({
    "1": "1st",
    "2": "2nd",
    "1r": "1st Refinanced",
    "2r": "2nd Refinanced"
}).fillna("")

In [433]:
#Format Owner Occupied

pr.rename(columns={"Owner Occ?": "oldOwnerOcc"}, inplace=True)

pr["Owner Occupied"] = np.where(
    pd.to_numeric(pr["oldOwnerOcc"], errors="coerce").fillna(0) == 0,
    "No",
    "Yes"
)

In [434]:
#Format Listed for Sale

pr.rename(columns={"Listed for Sale?": "oldListed"}, inplace=True)

pr["Listed for sale"] = np.where(pr["oldListed"] == 0, "No", "Yes")

In [435]:
#Add Auction Status

pr["Auction Status"] = np.where(
    pr["Sale Date"].isna() | (pr["Sale Date"] == ""),
    "Cancelled",
    "Pending"
)

In [436]:
#Format all dates

pr["Auction Date Updated"] = pd.to_datetime(
    pr["Sale Date"], errors="coerce"
).dt.strftime("%-m/%-d/%Y")

pr["Deed Date"] = pd.to_datetime(
    pr["Purchase Date"], errors="coerce"
).dt.strftime("%-m/%-d/%Y")

pr["Auction Date Original"] = pd.to_datetime(
    pr["Orig Sale Date"], errors="coerce"
).dt.strftime("%-m/%-d/%Y")

pr.rename(columns={
    "Sale Date": "oldSaleDate",
    "Purchase Date": "oldPurchaseDate",
    "Orig Sale Date": "oldOriginalSaleDate"
}, inplace=True)

In [437]:
#Remove Phoenix

pr = pr[pr["Market"] != "Phoenix"].copy()

In [438]:
#Remove Preforeclosure

pr = pr[pr["FCL Stage"] != "Preforeclosure"].copy()

In [439]:
#Rename columns

pr.rename(columns={
    "Sale Time": "Auction Sale Time",
    "Sale Place": "Auction Location",
    "TS Number": "NS Trustee Sale Number",
    "Trustee": "Auction Trustee",
    "Sq Ft": "Bldg Sq Ft",
    "Type": "Type of Property",
    "Yr Built": "Year Built",
    "FCL Lender": "Beneficiary",
    "FCL Loan Amt": "Mortgage",
    "Purchase Amt": "Deed",
    "Opening Bid $": "notOpeningBid",
    "Est Value": "RPR"
}, inplace=True)

In [440]:
#Remove unnecessary .0 from numeric columns

numeric_columns = [
    "ZIP",
    "Lot SqFt",
    "Foreclosure?",
    "RPR",
    "Year Built",
    "Bldg Sq Ft",
    "Mortgage",
    "Deed",
    "Beds",
    "Baths"
]

for col in numeric_columns:
    if col in pr.columns:
        pr[col] = (
            pr[col]
            .astype("string")
            .str.replace(r"\.0$", "", regex=True)
        )

In [441]:
#Format Auction Sale Time

pr["Auction Sale Time"] = (
    pr["Auction Sale Time"]
    .astype("string")
    .str.strip()
    .str.replace(r"^(\d{1,2}:\d{2}):\d{2}$", r"\1", regex=True)
)

In [442]:
#Remove extra row

pr = pr[
    pr["Last Name"].notna() &
    pr["Last Name"].astype(str).str.strip().ne("")
]

## Auction Trustee Lookup

In [443]:
#Find Auction Trustee Contact records

contacts = pd.read_csv('sf_data/trusteecontacts.csv')

In [444]:
#Normalize text for Contact Name and Associated Names

def norm(s):
    if pd.isna(s):
        return ""
    s = str(s).upper()
    s = s.replace("\u00A0", " ")              # non-breaking spaces
    s = re.sub(r"[^\w\s]", " ", s)            # remove punctuation
    s = re.sub(r"\s+", " ", s).strip()        # collapse spaces
    return s

contacts["contact_name_norm"] = contacts["Contact: Contact Name"].map(norm)
contacts["assoc_norm"] = contacts["Associated Names"].map(norm)  # can be blank

In [445]:
#Normalize trustee names

pr["trustee_norm"] = pr["Auction Trustee"].map(norm)

In [446]:
#Find exact matches

exact_map = dict(zip(contacts["contact_name_norm"], contacts["Contact: Contact Name"]))
pr["Auction Trustee (Lookup)"] = pr["trustee_norm"].map(exact_map)

In [447]:
#Find contains matches

def contains_in_contact_name(trustee_norm_value: str):
    if not trustee_norm_value:
        return None

    # find first contact where contact name contains trustee value
    hits = contacts.loc[
        contacts["contact_name_norm"].str.contains(re.escape(trustee_norm_value), na=False),
        "Contact: Contact Name"
    ]
    return hits.iloc[0] if len(hits) else None

mask = pr["Auction Trustee (Lookup)"].isna() & (pr["trustee_norm"] != "")
pr.loc[mask, "Auction Trustee (Lookup)"] = pr.loc[mask, "trustee_norm"].map(contains_in_contact_name)

In [448]:
#Find contains matches in associated names

def contains_in_associated(trustee_norm_value: str):
    if not trustee_norm_value:
        return None

    hits = contacts.loc[
        contacts["assoc_norm"].str.contains(re.escape(trustee_norm_value), na=False),
        "Contact: Contact Name"
    ]
    return hits.iloc[0] if len(hits) else None

mask = pr["Auction Trustee (Lookup)"].isna() & (pr["trustee_norm"] != "")
pr.loc[mask, "Auction Trustee (Lookup)"] = pr.loc[mask, "trustee_norm"].map(contains_in_associated)

In [449]:
#Collect and show unmatched rows

unmatched = pr.loc[
    pr["Auction Trustee (Lookup)"].isna() &
    pr["Auction Trustee"].notna() &
    pr["Auction Trustee"].astype(str).str.strip().ne(""),
    ["Last Name", "Auction Trustee", "trustee_norm"]
]

unmatched

,Last Name,Auction Trustee,trustee_norm
43,"2795 Twin Leaf Cir, Camp Verde, AZ 86322",TIMOTHY P REMICK,TIMOTHY P REMICK


In [450]:
#Confirm what should match but didnt

missing = (
    pr.loc[pr["Auction Trustee (Lookup)"].isna(), "trustee_norm"]
    .dropna().drop_duplicates()
)

# which of these are present as a normalized contact name?
present_mask = missing.isin(contacts["contact_name_norm"])
missing_not_in_contacts = missing[~present_mask]

missing_not_in_contacts.head(50)

43    TIMOTHY P REMICK
Name: trustee_norm, dtype: object

## Auction Location Lookup

In [451]:
locations = pd.read_csv('sf_data/auctionlocations.csv')

In [452]:
location_col = "Auction Location: Auction Location"

def normalize_location(value):
    if pd.isna(value):
        return ""

    value = str(value).upper().strip()

    # Correct known spelling differences
    value = value.replace("FRANSISCO", "FRANCISCO")

    # Standardize words
    replacements = {
        r"\bEAST\b": "E",
        r"\bWEST\b": "W",
        r"\bNORTH\b": "N",
        r"\bSOUTH\b": "S",
        r"\bSTREET\b": "ST",
        r"\bAVENUE\b": "AVE",
        r"\bROAD\b": "RD",
        r"\bDRIVE\b": "DR",
        r"\bBOULEVARD\b": "BLVD",
        r"\bSUITE\b": "STE",
        r"\bSTE\b": "UNIT",
        r"\bARIZONA\b": "AZ",
    }

    for pattern, replacement in replacements.items():
        value = re.sub(pattern, replacement, value)

    # Standardize suite numbers
    value = re.sub(r"#\s*(\w+)", r"UNIT \1", value)

    # Remove punctuation
    value = re.sub(r"[^\w\s]", " ", value)

    # Remove AZ and ZIP codes
    value = re.sub(r"\bAZ\b", " ", value)
    value = re.sub(r"\b\d{5}(?:\d{4})?\b", " ", value)

    # Remove common street suffixes
    value = re.sub(
        r"\b(ST|RD|DR|AVE|BLVD)\b", " ", value
    )

    # Clean spaces
    value = re.sub(r"\s+", " ", value)

    return value.strip()


# Prepare Salesforce locations
sf_locations = locations[location_col].dropna().unique()

location_lookup = [
    (name, normalize_location(name))
    for name in sf_locations
]


def get_street_number(value):
    match = re.match(r"^(\d+)\b", value)
    return match.group(1) if match else None


def match_location(value):
    normalized = normalize_location(value)

    if not normalized:
        return pd.Series([None, None])

    street_number = get_street_number(normalized)

    matches = []

    for sf_name, sf_normalized in location_lookup:

        sf_number = get_street_number(sf_normalized)

        # Reject different street numbers
        if street_number and sf_number:
            if street_number != sf_number:
                continue

        # Compare similarity
        score = fuzz.ratio(normalized, sf_normalized)

        matches.append((sf_name, score))

    if not matches:
        return pd.Series([None, None])

    # Select highest scoring match
    best_name, best_score = max(
        matches,
        key=lambda x: x[1]
    )

    # Require at least 85% similarity
    if best_score < 85:
        return pd.Series([None, round(best_score, 2)])

    return pd.Series([best_name, round(best_score, 2)])


# Create lookup and confidence columns
pr[["Auction Location Lookup", "Location Match %"]] = (
    pr["Auction Location"].apply(match_location)
)

In [453]:
review_locations = pr.loc[
    pr["Auction Location"].notna() &
    (
        pr["Auction Location Lookup"].isna() |
        (pr["Location Match %"] < 95)
    ),
    [
        "Auction Location",
        "Auction Location Lookup",
        "Location Match %"
    ]
].drop_duplicates().sort_values("Location Match %")

review_locations

,Auction Location,Auction Location Lookup,Location Match %
43,"661 E HOWARDS RD, CAMP VERDE",NaN,35.00
57,"OLD YAVAPAI COUNTY COURTHOUSE, PRESCOTT",NaN,67.65
30,"401 E SPRING ST, KINGMAN","401 SPRING STREET, KINGMAN, AZ 86401",94.74


In [454]:
# Manually correct verified location matches
manual_matches = {
    "971 N JASON LOPEZ CIR, FLORENCE":
        "971 JASON LOPEZ CIR #A - FLORENCE - AZ",

    "401 E SPRING ST, KINGMAN":
        "401 SPRING STREET, KINGMAN, AZ 86401"
}

# Apply manual corrections
pr["Auction Location Lookup"] = (
    pr["Auction Location"]
    .map(manual_matches)
    .fillna(pr["Auction Location Lookup"])
)

In [455]:
unmatched_locations = pr.loc[
    pr["Auction Location Lookup"].isna() &
    pr["Auction Location"].notna(),
    ["Auction Location", "Auction Location Lookup", "Location Match %"]
].drop_duplicates()

unmatched_locations

,Auction Location,Auction Location Lookup,Location Match %
43,"661 E HOWARDS RD, CAMP VERDE",NaN,35.00
57,"OLD YAVAPAI COUNTY COURTHOUSE, PRESCOTT",NaN,67.65


## Final Export

In [456]:
pr.to_csv("export_data/pr_import.csv", index=False)